# WarpTracer — vehicle controls

One box car, one floor, four walls. Run the three cells below to accelerate,
brake, and steer using invisible tire contact points. CPU is supported;
a GPU runtime uses CUDA when available. The first run compiles Warp kernels.

Setup reuses `/content/WarpTracer-rigidbody` and its single root uv environment,
switches to `vehicle-controls`, and fast-forward updates that branch.
The replay lets you orbit, zoom, pause, and scrub recorded motion.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "vehicle-controls", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "vehicle-controls:refs/remotes/origin/vehicle-controls"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/vehicle-controls"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "vehicle-controls", "origin/vehicle-controls"], check=True)
subprocess.run(["git", "switch", "vehicle-controls"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "vehicle-controls"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked", "--extra", "viz"], check=True)


## Choose a driving check

- `"accelerate-brake"`: accelerate straight, then brake to a stop.
- `"circle"`: drive with a constant left steering angle.
- `"s-turn"`: alternate left/right steering, then stop.

Change the scenario and rerun the final two cells. The existing `"drop"` and
`"wall-impact"` checks are also available. Physics runs at 240 Hz; the viewer
records 30 poses per second. Tire support currently assumes a flat floor.


In [ ]:
scenario = "accelerate-brake"  # "circle", "s-turn", "drop", or "wall-impact"
subprocess.run(["uv", "run", "--locked", "--extra", "viz", "warptracer-demo",
                "--scenario", scenario], check=True)


In [ ]:
from html import escape
from IPython.display import HTML, display

replay = Path("outputs") / f"{scenario}.html"
display(HTML(f'<iframe srcdoc="{escape(replay.read_text(), quote=True)}" '
             'width="100%" height="650" style="border:0"></iframe>'))
